# Teks — Laporan model (skor, kesalahan, keandalan, antar model; bahan slide aman NDA)
**Kapan dipakai:** setelah `baseline_oof` / `finetune_oof` / `ensemble_oof` sudah menyimpan OOF,
untuk menjelaskan di presentasi: model mana yang terbaik, di kelas mana salahnya, seberapa
bisa dipercaya probabilitasnya, dan apakah menggabung model membantu.

**Hanya membaca** hasil di `OOF_FOLDER` (tanpa melatih, tanpa GPU). Semua keluaran berupa
agregat: tidak ada teks, id, atau prediksi per baris yang dicetak atau disimpan.

**Langkah:**
1. Tabel percobaan (`percobaan.csv`) diurut menurut skor.
2. Skor tiap model dihitung ulang dari `oof_<nama>.npy` (F1 makro, akurasi).
3. Model terpilih: laporan per kelas, confusion matrix, pasangan kelas paling tertukar.
4. Kesalahan agregat: akurasi menurut panjang teks, kurva keandalan dan ECE.
5. Antar model: kesepakatan prediksi, rata-rata sederhana semua model vs model tunggal terbaik.
6. Sebaran prediksi test vs sebaran label train.

**Keluaran** di `OUTPUT_FOLDER`: PNG + tabel ringkas (CSV) + `fakta_model.md` (boleh dibawa
untuk slide).

**Data demo:** SmSA dengan OOF di `../outputs/text/oof`. Model yang bentuknya tidak cocok
dengan data latih (mis. dari soal lain) dilewati dan disebut.

In [ ]:
import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.metrics import precision_recall_fscore_support

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"                      # GANTI: None = tanpa sebaran test
OOF_FOLDER = "../../../outputs/text/oof"                       # oof_<nama>.npy, test_<nama>.npy
OUTPUT_FOLDER = "../../../outputs/text/laporan_model"

TEXT_COLS = ["text"]             # GANTI saat lomba: hanya untuk menghitung panjang teks
TARGET = "label"                 # GANTI saat lomba
MODEL_NAMES = None               # None = semua oof_*.npy yang cocok, atau ["tfidf_svc", ...]
FINAL_MODEL = None               # None = F1 makro OOF tertinggi, atau nama model
N_PAIRS = 3                      # pasangan kelas paling tertukar yang dilaporkan
AMAN_NDA = True                  # True = bin keandalan berisi < MIN_BIN baris tidak dilaporkan
MIN_BIN = 5
COLORS = ["#4c78a8", "#f58518", "#54a24b"]

## Alat simpan
Tiap grafik disimpan sebagai PNG + tabel ringkasnya; tiap temuan dicatat untuk `fakta_model.md`.

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False})
facts = {}
saved = []


def add_fact(section, text):
    facts.setdefault(section, []).append(text)
    print("-", text)


def save_figure(figure, name, table):
    figure.savefig(f"{OUTPUT_FOLDER}/{name}.png")
    table.to_csv(f"{OUTPUT_FOLDER}/{name}.csv", index=False)
    saved.append((name, table))
    plt.show()


def pct(value):
    return f"{100 * value:.1f}%".replace(".", ",")


print("alat simpan siap:", OUTPUT_FOLDER, "| AMAN_NDA:", AMAN_NDA)

## Baca data
Hanya label dan panjang teks (jumlah kata) yang dipakai; teksnya tidak dicetak.

In [ ]:
train = pd.read_csv(TRAIN_CSV)
classes = np.unique(train[TARGET])
class_names = classes.astype(str)
y = np.searchsorted(classes, train[TARGET].to_numpy())
n_classes = len(classes)
n_test = None
if TEST_CSV is not None:
    n_test = len(pd.read_csv(TEST_CSV))
raw_text = train[TEXT_COLS[0]].fillna("").astype(str)
for column in TEXT_COLS[1:]:
    raw_text = raw_text + " " + train[column].fillna("").astype(str)
n_words = raw_text.str.split().str.len()
base_share = np.bincount(y, minlength=n_classes) / len(y)
add_fact("Data", f"{len(train)} baris latih, {n_classes} kelas (kolom {TARGET}), "
         f"panjang teks median {n_words.median():.0f} kata.")
for index in range(n_classes):
    add_fact("Data", f"Kelas {class_names[index]}: {pct(base_share[index])} dari data latih.")

## 1. Tabel percobaan
Dari `percobaan.csv` di `OOF_FOLDER` (dicatat otomatis oleh notebook latih). Per
notebook + model hanya run terakhir yang dipakai; skor selain F1 (mis. RMSE) tidak digambar
karena arahnya berlawanan. Log bisa memuat soal lain: baca kolom `model`.

In [ ]:
LOG_COLUMNS = ["notebook", "model", "metrik", "skor_cv", "detik"]
log_path = f"{OOF_FOLDER}/percobaan.csv"
log = pd.DataFrame(columns=LOG_COLUMNS)
n_runs = 0
n_other = 0
if os.path.exists(log_path):
    full = pd.read_csv(log_path)
    n_runs = len(full)
    full = full.drop_duplicates(["notebook", "model"], keep="last")
    is_f1 = full["metrik"].astype(str).str.startswith("f1")
    n_other = int((~is_f1).sum())
    log = full[is_f1].sort_values("skor_cv", ascending=False)
    log = log[LOG_COLUMNS]
print(f"{n_runs} baris log, {len(log)} model F1 unik, {n_other} non-F1 dilewati")

In [ ]:
if len(log) > 0:
    top = log.head(15)
    first_row = top.iloc[0]
    labels = top["model"] + " (" + top["notebook"] + ")"
    figure, axis = plt.subplots(figsize=(9, 0.45 * len(top) + 1.2))
    axis.barh(labels, top["skor_cv"], color=COLORS[0])
    axis.invert_yaxis()
    axis.set_xlabel("skor CV (F1 makro)")
    for position, value in enumerate(top["skor_cv"]):
        axis.text(value, position, f" {value:.4f}", va="center", fontsize=8)
    add_fact("Percobaan", f"{n_runs} percobaan tercatat, {len(log)} model F1 unik. Terbaik: "
             f"{first_row['model']} ({first_row['notebook']}) F1 makro {first_row['skor_cv']:.4f}.")
    save_figure(figure, "01_percobaan", log)
else:
    print("percobaan.csv tidak ada atau tanpa baris F1: bagian ini dilewati")

## Muat OOF
Urutan kolom = kelas terurut (`np.unique`), sama seperti notebook latih. OOF yang jumlah
barisnya bukan jumlah data latih, jumlah kolomnya bukan jumlah kelas, atau belum lengkap
dilewati. `test_<nama>.npy` dipakai hanya bila bentuknya cocok.

In [ ]:
def read_oof(name):
    oof = np.load(f"{OOF_FOLDER}/oof_{name}.npy").astype("float64")
    if oof.shape != (len(train), n_classes):
        return f"bentuk {oof.shape}, perlu {(len(train), n_classes)}", None
    if not np.isfinite(oof).all():
        return "ada NaN/inf", None
    sums = oof.sum(axis=1, keepdims=True)
    if (sums <= 0).any():
        return "OOF belum lengkap (ada baris nol)", None
    return "", oof / sums


def read_test(name):
    path = f"{OOF_FOLDER}/test_{name}.npy"
    if not os.path.exists(path):
        return None
    test_pred = np.load(path).astype("float64")
    if test_pred.ndim != 2 or test_pred.shape[1] != n_classes:
        return None
    if n_test is not None and len(test_pred) != n_test:
        return None
    return test_pred


oofs, tests, skipped = {}, {}, {}

In [ ]:
for path in sorted(glob.glob(f"{OOF_FOLDER}/oof_*.npy")):
    name = os.path.basename(path)[4:-4]
    if MODEL_NAMES is not None and name not in MODEL_NAMES:
        continue
    reason, oof = read_oof(name)
    if reason:
        skipped[name] = reason
        continue
    oofs[name] = oof
    tests[name] = read_test(name)
if len(oofs) == 0:
    raise ValueError("Tidak ada OOF yang cocok: jalankan baseline_oof dulu / cek TRAIN_CSV, TARGET")
print("terbaca:", list(oofs))
for name, reason in skipped.items():
    print(f"  dilewati {name}: {reason}")
add_fact("Data", f"{len(oofs)} model terbaca: {', '.join(oofs)}. {len(skipped)} dilewati "
         f"karena bentuk tidak cocok: {', '.join(skipped) or '-'}.")

## 2. Skor OOF tiap model
Dihitung ulang dari `.npy` (argmax probabilitas), jadi tidak bergantung pada angka di log.

In [ ]:
preds = {}
score_rows = []
for name, oof in oofs.items():
    preds[name] = oof.argmax(axis=1)
    score_rows.append({"model": name, "f1_makro": f1_score(y, preds[name], average="macro"),
                       "akurasi": accuracy_score(y, preds[name]),
                       "ada_test": tests[name] is not None})
scores = pd.DataFrame(score_rows)
scores = scores.sort_values("f1_makro", ascending=False).reset_index(drop=True)
chosen = scores.loc[0, "model"]
if FINAL_MODEL is not None:
    if FINAL_MODEL in oofs:
        chosen = FINAL_MODEL
    else:
        print("FINAL_MODEL tidak ada di model terbaca, pakai yang terbaik:", chosen)
print(scores.round(4).to_string(index=False))
print("model terpilih:", chosen)

In [ ]:
positions = np.arange(len(scores))
figure, axis = plt.subplots(figsize=(9, 0.6 * len(scores) + 1.5))
axis.barh(positions - 0.2, scores["f1_makro"], height=0.38, color=COLORS[0], label="F1 makro")
axis.barh(positions + 0.2, scores["akurasi"], height=0.38, color=COLORS[1], label="akurasi")
axis.set_yticks(positions)
axis.set_yticklabels(scores["model"])
axis.invert_yaxis()
axis.set_xlim(0, 1.08)
axis.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.2))
for position in range(len(scores)):
    axis.text(scores.loc[position, "f1_makro"], position - 0.2,
              f" {scores.loc[position, 'f1_makro']:.3f}", va="center", fontsize=8)
    axis.text(scores.loc[position, "akurasi"], position + 0.2,
              f" {scores.loc[position, 'akurasi']:.3f}", va="center", fontsize=8)
axis.set_xlabel("skor OOF")
best = scores.loc[0]
add_fact("Skor OOF", f"Terbaik: {best['model']} F1 makro {best['f1_makro']:.4f}, "
         f"akurasi {best['akurasi']:.4f} ({len(scores)} model dihitung ulang dari OOF).")
add_fact("Skor OOF", f"Model terpilih untuk bagian berikut: {chosen}.")
save_figure(figure, "02_skor_oof", scores)

## 3. Model terpilih: laporan per kelas

In [ ]:
probs = oofs[chosen]
pred = preds[chosen]
labels_all = np.arange(n_classes)
precision, recall, f1_class, support = precision_recall_fscore_support(
    y, pred, labels=labels_all, zero_division=0)
report = pd.DataFrame({"kelas": class_names, "precision": precision, "recall": recall,
                       "f1": f1_class, "jumlah": support})
rotation = 0
if n_classes > 6:
    rotation = 60
figure, axis = plt.subplots(figsize=(max(7, 0.7 * n_classes + 3), 4))
for step, column in enumerate(["precision", "recall", "f1"]):
    axis.bar(labels_all + (step - 1) * 0.27, report[column], width=0.27, color=COLORS[step],
             label=column)
axis.set_xticks(labels_all)
axis.set_xticklabels(class_names, rotation=rotation, ha="right")
axis.set_ylim(0, 1.05)
axis.legend(frameon=False, ncol=3, loc="lower right")
weak, strong = report["f1"].idxmin(), report["f1"].idxmax()
add_fact("Model terpilih", f"{chosen}: F1 terendah di kelas {report.loc[weak, 'kelas']} "
         f"({report.loc[weak, 'f1']:.3f}, {int(report.loc[weak, 'jumlah'])} contoh), "
         f"tertinggi di {report.loc[strong, 'kelas']} ({report.loc[strong, 'f1']:.3f}).")
save_figure(figure, "03_laporan_kelas", report.round(4))

## 3b. Confusion matrix
Warna = persen per baris (dari kelas sebenarnya); angka = jumlah dan persen baris.

In [ ]:
cm = confusion_matrix(y, pred, labels=labels_all)
row_pct = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
size = max(5, 0.55 * n_classes + 3)
figure, axis = plt.subplots(figsize=(size, size * 0.9))
image = axis.imshow(row_pct, cmap="Blues", vmin=0, vmax=1)
figure.colorbar(image, ax=axis, label="porsi baris")
axis.set_xticks(labels_all)
axis.set_yticks(labels_all)
axis.set_xticklabels(class_names, rotation=45, ha="right")
axis.set_yticklabels(class_names)
axis.set_xlabel("prediksi")
axis.set_ylabel("sebenarnya")
if n_classes <= 12:
    for row in range(n_classes):
        for col in range(n_classes):
            color = "black"
            if row_pct[row, col] > 0.5:
                color = "white"
            axis.text(col, row, f"{cm[row, col]}\n{pct(row_pct[row, col])}", ha="center",
                      va="center", color=color, fontsize=9)

In [ ]:
cm_table = pd.DataFrame(cm, columns="pred_" + class_names)
cm_table.insert(0, "sebenarnya", class_names)
n_wrong = int(cm.sum() - np.trace(cm))
add_fact("Model terpilih", f"Confusion matrix {chosen}: {int(np.trace(cm))} benar, {n_wrong} salah "
         f"dari {int(cm.sum())} (akurasi {pct(np.trace(cm) / cm.sum())}).")
save_figure(figure, "04_confusion_matrix", cm_table)

## 3c. Pasangan kelas paling sering tertukar
Dua arah dijumlahkan (A diprediksi B ditambah B diprediksi A); A dan B = kiri dan kanan tanda <->.

In [ ]:
def confused_pairs(cm, names, top_n):
    rows = []
    for first in range(len(names)):
        for second in range(first + 1, len(names)):
            swapped = cm[first, second] + cm[second, first]
            rows.append({"kelas_a": names[first], "kelas_b": names[second],
                         "a_jadi_b": cm[first, second], "b_jadi_a": cm[second, first],
                         "total_tertukar": swapped})
    pairs = pd.DataFrame(rows).sort_values("total_tertukar", ascending=False)
    return pairs.head(top_n).reset_index(drop=True)


pairs = confused_pairs(cm, class_names, N_PAIRS)
pair_labels = pairs["kelas_a"] + " <-> " + pairs["kelas_b"]
figure, axis = plt.subplots(figsize=(8, 0.6 * len(pairs) + 1.5))
axis.barh(pair_labels, pairs["a_jadi_b"], color=COLORS[0], label="kelas A diprediksi B")
axis.barh(pair_labels, pairs["b_jadi_a"], left=pairs["a_jadi_b"], color=COLORS[1],
          label="kelas B diprediksi A")
axis.invert_yaxis()
axis.set_xlabel("jumlah baris tertukar")
axis.legend(frameon=False)
for row in range(len(pairs)):
    kelas_a, kelas_b = pairs.loc[row, "kelas_a"], pairs.loc[row, "kelas_b"]
    add_fact("Kelas tertukar", f"{kelas_a} vs {kelas_b}: {pairs.loc[row, 'a_jadi_b']} baris "
             f"{kelas_a} diprediksi {kelas_b}, {pairs.loc[row, 'b_jadi_a']} sebaliknya = "
             f"{pct(pairs.loc[row, 'total_tertukar'] / max(n_wrong, 1))} dari semua salah.")
save_figure(figure, "05_pasangan_tertukar", pairs)

## 4. Kesalahan agregat: panjang teks
Data latih dibagi 4 sama banyak menurut jumlah kata (K1 = paling pendek).

In [ ]:
quartile = pd.qcut(n_words.rank(method="first"), 4, labels=["K1", "K2", "K3", "K4"])
correct = pred == y
length_rows = []
for label in quartile.cat.categories:
    mask = (quartile == label).to_numpy()
    length_rows.append({"kuartil": label, "kata_min": int(n_words[mask].min()),
                        "kata_maks": int(n_words[mask].max()), "jumlah": int(mask.sum()),
                        "akurasi": correct[mask].mean(),
                        "f1_makro": f1_score(y[mask], pred[mask], average="macro")})
length_table = pd.DataFrame(length_rows)
tick_labels = length_table["kuartil"] + "\n" + length_table["kata_min"].astype(str)
tick_labels = tick_labels + "-" + length_table["kata_maks"].astype(str) + " kata"
figure, axis = plt.subplots(figsize=(7, 4))
axis.bar(tick_labels, length_table["akurasi"], color=COLORS[0])
axis.set_ylabel("akurasi OOF")
axis.set_ylim(0, 1.05)
for position, value in enumerate(length_table["akurasi"]):
    axis.text(position, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)
low, high = length_table["akurasi"].idxmin(), length_table["akurasi"].idxmax()
add_fact("Panjang teks", f"{chosen}: akurasi terendah di {length_table.loc[low, 'kuartil']} "
         f"({length_table.loc[low, 'kata_min']}-{length_table.loc[low, 'kata_maks']} kata, "
         f"{length_table.loc[low, 'akurasi']:.3f}), tertinggi di "
         f"{length_table.loc[high, 'kuartil']} ({length_table.loc[high, 'akurasi']:.3f}).")
save_figure(figure, "06_akurasi_panjang_teks", length_table.round(4))

## 4b. Kurva keandalan dan ECE
Keyakinan = probabilitas kelas tertinggi. Model andal bila akurasi tiap bin dekat keyakinannya
(titik di garis diagonal). ECE = rata-rata selisih itu, dibobot jumlah baris (0 = sempurna).
ECE dihitung dari semua baris; dengan `AMAN_NDA` bin berisi kurang dari `MIN_BIN` baris tidak
ditampilkan.

In [ ]:
def reliability_table(confidence, correct):
    edges = np.linspace(0, 1, 11)
    bins = np.clip(np.digitize(confidence, edges[1:-1]), 0, 9)
    rows = []
    for index in range(10):
        mask = bins == index
        if mask.sum() > 0:
            rows.append({"bin": f"{edges[index]:.1f}-{edges[index + 1]:.1f}",
                         "tengah": (edges[index] + edges[index + 1]) / 2,
                         "jumlah": int(mask.sum()), "keyakinan": confidence[mask].mean(),
                         "akurasi": correct[mask].mean()})
    return pd.DataFrame(rows)


confidence = probs.max(axis=1)
full_bins = reliability_table(confidence, correct)
gap = (full_bins["akurasi"] - full_bins["keyakinan"]).abs()
ece = (full_bins["jumlah"] / len(y) * gap).sum()
shown = full_bins
if AMAN_NDA:
    shown = full_bins[full_bins["jumlah"] >= MIN_BIN]
print(f"ECE {ece:.4f} | bin penuh {len(full_bins)}, ditampilkan {len(shown)}")

In [ ]:
figure, axis = plt.subplots(figsize=(6, 5.5))
axis.plot([0, 1], [0, 1], "--", color="gray", label="andal sempurna")
axis.bar(shown["tengah"], shown["akurasi"], width=0.09, color=COLORS[0], alpha=0.8,
         label="akurasi bin")
axis.plot(shown["keyakinan"], shown["akurasi"], "o", color=COLORS[1], label="keyakinan rata-rata")
axis.set_xlabel("keyakinan (probabilitas tertinggi)")
axis.set_ylabel("akurasi")
axis.set_xlim(0, 1)
axis.set_ylim(0, 1.05)
axis.legend(frameon=False, loc="upper left")
mean_conf = confidence.mean()
direction = "terlalu hati-hati"
if mean_conf > correct.mean():
    direction = "terlalu percaya diri"
add_fact("Keandalan", f"{chosen}: ECE {ece:.4f}, keyakinan rata-rata {mean_conf:.3f} vs akurasi "
         f"{correct.mean():.3f} ({direction}). {pct((confidence >= 0.9).mean())} baris punya "
         f"keyakinan >= 0,9 dengan akurasi {correct[confidence >= 0.9].mean():.3f}.")
save_figure(figure, "07_kurva_keandalan", shown.round(4))

## 5. Antar model: kesepakatan prediksi
Porsi baris yang diprediksi sama oleh dua model. Rendah = model saling melengkapi (gabungan
berpotensi naik); tinggi = hampir kembar.

In [ ]:
names = list(oofs)
agreement = np.ones((len(names), len(names)))
for first in range(len(names)):
    for second in range(len(names)):
        agreement[first, second] = (preds[names[first]] == preds[names[second]]).mean()
if len(names) >= 2:
    size = max(4.5, 0.9 * len(names) + 2.5)
    figure, axis = plt.subplots(figsize=(size, size * 0.85))
    image = axis.imshow(agreement, cmap="Blues", vmin=agreement.min(), vmax=1)
    figure.colorbar(image, ax=axis, label="porsi prediksi sama")
    axis.set_xticks(range(len(names)))
    axis.set_yticks(range(len(names)))
    axis.set_xticklabels(names, rotation=45, ha="right")
    axis.set_yticklabels(names)
    for row in range(len(names)):
        for col in range(len(names)):
            axis.text(col, row, f"{agreement[row, col]:.2f}", ha="center", va="center",
                      fontsize=9)
else:
    print("hanya 1 model terbaca: kesepakatan antar model dilewati")

In [ ]:
if len(names) >= 2:
    off_diagonal = agreement.copy()
    np.fill_diagonal(off_diagonal, np.nan)
    least = np.unravel_index(np.nanargmin(off_diagonal), off_diagonal.shape)
    most = np.unravel_index(np.nanargmax(off_diagonal), off_diagonal.shape)
    add_fact("Antar model", f"Kesepakatan rata-rata {np.nanmean(off_diagonal):.3f}, paling "
             f"beda {names[least[0]]} vs {names[least[1]]} ({off_diagonal[least]:.3f}), paling "
             f"mirip {names[most[0]]} vs {names[most[1]]} ({off_diagonal[most]:.3f}).")
    save_figure(figure, "08_kesepakatan_model", pd.DataFrame(agreement.round(4), columns=names))

## 5b. Rata-rata probabilitas semua model vs model tunggal terbaik
Rata-rata sederhana tanpa bobot (tidak ada yang dilatih, jadi tidak bocor). Hasil ini
patokan: bobot belajar di `ensemble_oof` baru bermakna bila mengalahkannya.

In [ ]:
if len(names) >= 2:
    mean_pred = np.mean(list(oofs.values()), axis=0).argmax(axis=1)
    compare = pd.DataFrame({
        "cara": [f"tunggal terbaik ({best['model']})", f"rata-rata {len(names)} model"],
        "f1_makro": [best["f1_makro"], f1_score(y, mean_pred, average="macro")],
        "akurasi": [best["akurasi"], accuracy_score(y, mean_pred)]})
    figure, axis = plt.subplots(figsize=(6.5, 4))
    for step, column in enumerate(["f1_makro", "akurasi"]):
        axis.bar(np.arange(2) + (step - 0.5) * 0.35, compare[column], width=0.35,
                 color=COLORS[step], label=column)
        for position, value in enumerate(compare[column]):
            axis.text(position + (step - 0.5) * 0.35, value, f"{value:.3f}", ha="center",
                      va="bottom", fontsize=9)
    axis.set_xticks(range(2))
    axis.set_xticklabels(compare["cara"])
    axis.set_ylim(0, 1.08)
    axis.legend(frameon=False, loc="lower right")
    delta = compare.loc[1, "f1_makro"] - compare.loc[0, "f1_makro"]
    add_fact("Antar model", f"Rata-rata sederhana {len(names)} model: F1 makro "
             f"{compare.loc[1, 'f1_makro']:.4f} vs tunggal terbaik "
             f"{compare.loc[0, 'f1_makro']:.4f} "
             f"(selisih {delta:+.4f}). Model lemah ikut merata-ratakan, jadi ini bukan batas atas.")
    save_figure(figure, "09_rata_rata_vs_tunggal", compare.round(4))
else:
    print("hanya 1 model terbaca: perbandingan rata-rata dilewati")

## 6. Sebaran prediksi test vs label train
Prediksi test = argmax `test_<model>.npy` (tanpa pengali kelas). Sebaran prediksi OOF ditaruh
sebagai pembanding: selisih test-vs-train yang sama besar dengan OOF-vs-train berarti
kecenderungan model, bukan pergeseran data.

In [ ]:
test_proba = tests[chosen]
if test_proba is None:
    print(f"test_{chosen}.npy tidak ada atau bentuknya tidak cocok: bagian ini dilewati")
else:
    test_share = np.bincount(test_proba.argmax(axis=1), minlength=n_classes) / len(test_proba)
    oof_share = np.bincount(pred, minlength=n_classes) / len(pred)
    shift = pd.DataFrame({"kelas": class_names, "porsi_label_train": base_share,
                          "porsi_prediksi_oof": oof_share, "porsi_prediksi_test": test_share})
    figure, axis = plt.subplots(figsize=(max(7, 0.7 * n_classes + 3), 4))
    for step, column in enumerate(shift.columns[1:]):
        axis.bar(labels_all + (step - 1) * 0.27, shift[column], width=0.27, color=COLORS[step],
                 label=column.replace("porsi_", "").replace("_", " "))
    axis.set_xticks(labels_all)
    axis.set_xticklabels(class_names, rotation=rotation, ha="right")
    axis.set_ylabel("porsi baris")
    axis.legend(frameon=False)
    diff = test_share - base_share
    big = np.abs(diff).argmax()
    add_fact("Sebaran test", f"{chosen}: prediksi test {len(test_proba)} baris. Selisih terbesar "
             f"dari label train di kelas {class_names[big]} ({pct(test_share[big])} vs "
             f"{pct(base_share[big])}), jarak total {pct(np.abs(diff).sum() / 2)} (0 = sama).")
    save_figure(figure, "10_sebaran_prediksi_test", shift.round(4))

## Fakta untuk slide
`fakta_model.md` = semua temuan + tabel di balik tiap grafik, tanpa teks asli. Isinya juga
dicetak di bawah supaya ikut tersimpan di output notebook.

In [ ]:
lines = ["# Fakta laporan model (otomatis dari laporan_model)", "",
         f"Dibuat {time.strftime('%Y-%m-%d %H:%M')}. Semua angka dihitung dari OOF di "
         f"{OOF_FOLDER}.",
         "Untuk AI pembuat slide: jangan menambah angka atau nama model yang tidak tertulis "
         "di sini.",
         "Hanya agregat: tidak ada teks, id, atau prediksi per baris.", ""]
for section, items in facts.items():
    lines.append(f"## {section}")
    for item in items:
        lines.append(f"- {item}")
    lines.append("")
lines.append("## Tabel di balik grafik")
for name, table in saved:
    lines.append(f"### {name}.png")
    lines.append("```")
    lines.append(table.head(40).to_string(index=False))
    lines.append("```")
with open(f"{OUTPUT_FOLDER}/fakta_model.md", "w", encoding="utf-8") as report_file:
    report_file.write("\n".join(lines))
print("\n".join(lines))